# Notebook 03 — Feature Engineering & Train/Test Split

- Create useful derived features from the actual dataset
- Exclude identifiers and prevent target leakage
- Separate features and target
- Create a stratified train/test split
- Fit preprocessing only on training data
- One-hot encode categorical features
- Scale numerical features
- Save the model-ready data and preprocessing artifacts


## 1. Imports, Variables, and Paths

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

target = "Job_Accepted"
RANDOM_STATE = 42
TEST_SIZE = 0.20

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_job_acceptance_data.csv"

if not DATA_PATH.exists():
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_job_acceptance_data.csv"

MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())
print("Target:", target)

Project root: D:\job acceptance prediction system
Dataset: D:\job acceptance prediction system\data\processed\cleaned_job_acceptance_data.csv
Dataset exists: True
Target: Job_Accepted


## 2. Load and Validate the Cleaned Dataset

In [2]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Target values:", sorted(df[target].unique().tolist()))

if target not in df.columns:
    raise KeyError(f"Missing target column: {target}")

if set(df[target].unique()) != {0, 1}:
    raise ValueError("Job_Accepted must contain exactly 0 and 1.")

if df.isna().sum().sum() != 0:
    raise ValueError("Cleaned dataset contains missing values.")

display(df.head())

Shape: (49860, 36)
Missing values: 0
Duplicate rows: 0
Target values: [0, 1]


,Candidate_ID,Age,Gender,Education_Level,Degree_Field,University_Tier,CGPA,Years_of_Experience,Previous_Companies,Internship_Experience,...,Relocation_Willingness,Competition_Level,Expected_Salary,Offered_Salary,Career_Growth_Score,Benefits_Score,Job_Security_Score,Notice_Period_Days,Offer_to_Joining_Days,Job_Accepted
0,CAND00001,28.0,Male,Diploma,Computer Science,Tier 2,7.60,5.2,3.0,No,...,Yes,Medium,1225000.0,1198000.0,92.1,62.0,77.5,30,54,0
1,CAND00002,22.0,Female,Bachelor's,Engineering,Tier 3,7.74,1.5,0.0,No,...,No,Medium,1029000.0,1168000.0,93.8,94.6,55.4,30,44,1
2,CAND00003,30.0,Female,Master's,Business Administration,Tier 1,8.28,5.1,3.0,Yes,...,Yes,Medium,644000.0,681000.0,84.4,65.9,66.3,30,31,0
3,CAND00004,31.0,Female,Bachelor's,Data Science,Tier 2,7.96,5.5,3.0,No,...,No,Low,1166000.0,1213000.0,87.3,85.6,79.7,45,58,1
4,CAND00005,20.0,Male,Bachelor's,Computer Science,Tier 3,7.96,0.0,0.0,No,...,Yes,Medium,597000.0,666000.0,83.7,76.4,79.3,15,37,1


## 3. Remove Identifier Columns

In [3]:
identifier_columns = [c for c in ["Candidate_ID"] if c in df.columns]

df_model = df.drop(columns=identifier_columns).copy()

print("Removed:", identifier_columns)
print("Shape after identifier removal:", df_model.shape)

Removed: ['Candidate_ID']
Shape after identifier removal: (49860, 35)


## 4. Feature Engineering

The derived features below are created from candidate/job information only.

- `Experience_Category`
- `Academic_Performance_Band`
- `Skills_Match_Level`
- `Interview_Performance_Category`
- `Salary_Gap`
- `Salary_Match_Percentage`

A `Placement_Probability_Score` is **not** created because it would overlap with the prediction task and could cause target leakage if calculated from the target.

In [4]:
# Experience category
df_model["Experience_Category"] = pd.cut(
    df_model["Years_of_Experience"],
    bins=[-np.inf, 1, 5, 10, np.inf],
    labels=["Fresher/Entry", "Junior", "Mid-Level", "Senior"]
)

# Academic performance category
df_model["Academic_Performance_Band"] = pd.cut(
    df_model["CGPA"],
    bins=[-np.inf, 6.49, 7.99, np.inf],
    labels=["Low", "Medium", "High"]
)

# Skills match category
df_model["Skills_Match_Level"] = pd.cut(
    df_model["Skills_Match_Percentage"],
    bins=[-np.inf, 49.99, 74.99, np.inf],
    labels=["Low", "Medium", "High"]
)

# Interview performance category
df_model["Interview_Performance_Category"] = pd.cut(
    df_model["Interview_Score"],
    bins=[-np.inf, 49.99, 74.99, np.inf],
    labels=["Low", "Medium", "High"]
)

# Salary-derived features
df_model["Salary_Gap"] = (
    df_model["Offered_Salary"] - df_model["Expected_Salary"]
)

df_model["Salary_Match_Percentage"] = np.where(
    df_model["Expected_Salary"] > 0,
    df_model["Offered_Salary"] / df_model["Expected_Salary"] * 100,
    np.nan
)

new_features = [c for c in df_model.columns if c not in df.columns]

print("New engineered features:")
for c in new_features:
    print("-", c)

print("\nNew shape:", df_model.shape)
display(df_model[new_features].head())

New engineered features:
- Experience_Category
- Academic_Performance_Band
- Skills_Match_Level
- Interview_Performance_Category
- Salary_Gap
- Salary_Match_Percentage

New shape: (49860, 41)


,Experience_Category,Academic_Performance_Band,Skills_Match_Level,Interview_Performance_Category,Salary_Gap,Salary_Match_Percentage
0,Mid-Level,Medium,Medium,High,-27000.0,97.795918
1,Junior,Medium,Low,High,139000.0,113.508260
2,Mid-Level,High,Medium,High,37000.0,105.745342
3,Mid-Level,Medium,Medium,High,47000.0,104.030875
4,Fresher/Entry,Medium,Low,Medium,69000.0,111.557789


## 5. Validate Engineered Features

In [5]:
display(df_model[new_features].describe(include="all").T)

print("Missing values introduced by feature engineering:")
display(df_model[new_features].isna().sum().to_frame("Missing_Values"))

# Expected salary was already cleaned, so Salary_Match_Percentage should be valid.
# Any unexpected missing value would be investigated before modeling.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Experience_Category,49860,4,Junior,24675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Academic_Performance_Band,49860,3,Medium,26063,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Skills_Match_Level,49860,3,Medium,31948,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Interview_Performance_Category,49860,3,High,25694,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Salary_Gap,49860.0,NaN,NaN,NaN,40260.733617,144014.587515,-2194682.8,-38000.0,39000.0,115000.0,2702748.25
Salary_Match_Percentage,49860.0,NaN,NaN,NaN,105.049941,16.608589,19.087954,95.399084,104.743804,113.797863,619.374831


Missing values introduced by feature engineering:


,Missing_Values
Experience_Category,0
Academic_Performance_Band,0
Skills_Match_Level,0
Interview_Performance_Category,0
Salary_Gap,0
Salary_Match_Percentage,0


## 6. Target Leakage Check

In [6]:
# Target must never be part of the input feature set.
X = df_model.drop(columns=[target]).copy()
y = df_model[target].copy()

leakage_keywords = [
    "accepted",
    "acceptance",
    "placement_probability",
    "prediction",
    "predicted"
]

potential_leakage = [
    c for c in X.columns
    if any(k in c.lower() for k in leakage_keywords)
]

print("Potential leakage-named features:", potential_leakage)

if target in X.columns:
    raise ValueError("Target leakage: Job_Accepted is present in X.")

print("Target is excluded from X.")

Potential leakage-named features: []
Target is excluded from X.


## 7. Identify Numerical and Categorical Features

In [7]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 21
['Age', 'CGPA', 'Years_of_Experience', 'Previous_Companies', 'Certifications_Count', 'Technical_Skills_Score', 'Soft_Skills_Score', 'Skills_Match_Percentage', 'Aptitude_Score', 'Communication_Score', 'Interview_Score', 'Interview_Rounds', 'Expected_Salary', 'Offered_Salary', 'Career_Growth_Score', 'Benefits_Score', 'Job_Security_Score', 'Notice_Period_Days', 'Offer_to_Joining_Days', 'Salary_Gap', 'Salary_Match_Percentage']

Categorical features: 19
['Gender', 'Education_Level', 'Degree_Field', 'University_Tier', 'Internship_Experience', 'Job_Role', 'Industry', 'Company_Tier', 'Company_Size', 'Job_Type', 'Work_Mode', 'Job_Location', 'Preferred_Location', 'Relocation_Willingness', 'Competition_Level', 'Experience_Category', 'Academic_Performance_Band', 'Skills_Match_Level', 'Interview_Performance_Category']


## 8. Stratified Train/Test Split

An 80/20 split is used with `stratify=y` so the accepted/rejected proportions remain approximately consistent in both sets.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

split_summary = pd.DataFrame({
    "Train_Count": y_train.value_counts().sort_index(),
    "Train_Percent": y_train.value_counts(normalize=True).sort_index().mul(100),
    "Test_Count": y_test.value_counts().sort_index(),
    "Test_Percent": y_test.value_counts(normalize=True).sort_index().mul(100)
})

display(split_summary.round(2))

X_train: (39888, 40)
X_test: (9972, 40)
y_train: (39888,)
y_test: (9972,)


,Train_Count,Train_Percent,Test_Count,Test_Percent
Job_Accepted,,,,
0,14420,36.15,3605,36.15
1,25468,63.85,6367,63.85


## 9. Build the Preprocessing Pipeline

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessor created.")

Preprocessor created.


## 10. Fit Preprocessor on Training Data Only

This is an important leakage-prevention step.

- `fit_transform()` → training data
- `transform()` → test data

The test set is never used to learn scaling parameters or category mappings.

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_test:", X_test_processed.shape)
print("Sparse representation:", sparse.issparse(X_train_processed))

Processed X_train: (39888, 102)
Processed X_test: (9972, 102)
Sparse representation: False


## 11. Inspect Final Model Features

In [11]:
feature_names = preprocessor.get_feature_names_out()

print("Final encoded feature count:", len(feature_names))

print("\nFirst 30 features:")
for feature in feature_names[:30]:
    print("-", feature)

print("\nLast 10 features:")
for feature in feature_names[-10:]:
    print("-", feature)

Final encoded feature count: 102

First 30 features:
- numeric__Age
- numeric__CGPA
- numeric__Years_of_Experience
- numeric__Previous_Companies
- numeric__Certifications_Count
- numeric__Technical_Skills_Score
- numeric__Soft_Skills_Score
- numeric__Skills_Match_Percentage
- numeric__Aptitude_Score
- numeric__Communication_Score
- numeric__Interview_Score
- numeric__Interview_Rounds
- numeric__Expected_Salary
- numeric__Offered_Salary
- numeric__Career_Growth_Score
- numeric__Benefits_Score
- numeric__Job_Security_Score
- numeric__Notice_Period_Days
- numeric__Offer_to_Joining_Days
- numeric__Salary_Gap
- numeric__Salary_Match_Percentage
- categorical__Gender_Female
- categorical__Gender_Male
- categorical__Gender_Other
- categorical__Education_Level_Bachelor's
- categorical__Education_Level_Diploma
- categorical__Education_Level_Master's
- categorical__Education_Level_PhD
- categorical__Degree_Field_Arts
- categorical__Degree_Field_Business Administration

Last 10 features:
- categor

## 12. Verify Processed Data

In [12]:
def contains_nan(matrix):
    if sparse.issparse(matrix):
        return bool(np.isnan(matrix.data).any())
    return bool(np.isnan(matrix).any())

train_nan = contains_nan(X_train_processed)
test_nan = contains_nan(X_test_processed)

print("NaN in training matrix:", train_nan)
print("NaN in test matrix:", test_nan)

if train_nan or test_nan:
    raise ValueError("NaN values remain after preprocessing.")

print("No NaN values remain.")

NaN in training matrix: False
NaN in test matrix: False
No NaN values remain.


## 13. Verify Train/Test Separation

In [13]:
overlap = set(X_train.index).intersection(set(X_test.index))

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Overlapping rows:", len(overlap))

if overlap:
    raise ValueError("Train/test overlap detected.")

print("No train/test overlap detected.")

Train rows: 39888
Test rows: 9972
Overlapping rows: 0
No train/test overlap detected.


## 14. Save Feature-Engineered Dataset

In [14]:
FEATURE_ENGINEERED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_engineered_job_acceptance_data.csv"
)

df_model.to_csv(FEATURE_ENGINEERED_PATH, index=False)

print("Saved:", FEATURE_ENGINEERED_PATH)
print("Shape:", df_model.shape)

Saved: D:\job acceptance prediction system\data\processed\feature_engineered_job_acceptance_data.csv
Shape: (49860, 41)


## 15. Save ML Artifacts

In [15]:
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor.joblib"
FEATURE_NAMES_PATH = MODEL_DIR / "feature_names.joblib"
TRAIN_TEST_PATH = MODEL_DIR / "train_test_data.joblib"

joblib.dump(preprocessor, PREPROCESSOR_PATH)
joblib.dump(feature_names.tolist(), FEATURE_NAMES_PATH)

joblib.dump(
    {
        "X_train": X_train_processed,
        "X_test": X_test_processed,
        "y_train": y_train.to_numpy(),
        "y_test": y_test.to_numpy()
    },
    TRAIN_TEST_PATH
)

print("Saved artifacts:")
print("-", PREPROCESSOR_PATH)
print("-", FEATURE_NAMES_PATH)
print("-", TRAIN_TEST_PATH)

Saved artifacts:
- D:\job acceptance prediction system\models\preprocessor.joblib
- D:\job acceptance prediction system\models\feature_names.joblib
- D:\job acceptance prediction system\models\train_test_data.joblib


## 16. Reload and Verify Saved Artifacts

In [16]:
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)
loaded_feature_names = joblib.load(FEATURE_NAMES_PATH)
loaded_data = joblib.load(TRAIN_TEST_PATH)

print("Loaded preprocessor:", type(loaded_preprocessor))
print("Loaded feature names:", len(loaded_feature_names))
print("Loaded X_train:", loaded_data["X_train"].shape)
print("Loaded X_test:", loaded_data["X_test"].shape)

assert loaded_data["X_train"].shape == X_train_processed.shape
assert loaded_data["X_test"].shape == X_test_processed.shape
assert len(loaded_feature_names) == X_train_processed.shape[1]

print("\nArtifact verification PASSED.")

Loaded preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Loaded feature names: 102
Loaded X_train: (39888, 102)
Loaded X_test: (9972, 102)

Artifact verification PASSED.


## 17. Final Validation

In [17]:
print("NOTEBOOK 03 VALIDATION")
print("=" * 70)

print(f"Original cleaned dataset: {df.shape}")
print(f"Feature-engineered dataset: {df_model.shape}")
print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")
print(f"Input model features before encoding: {X.shape[1]:,}")
print(f"Final encoded features: {len(feature_names):,}")
print(f"Training acceptance rate: {y_train.mean() * 100:.2f}%")
print(f"Testing acceptance rate: {y_test.mean() * 100:.2f}%")
print(f"Train/test overlap: {len(overlap)}")
print(f"Training matrix contains NaN: {train_nan}")
print(f"Test matrix contains NaN: {test_nan}")

print("\nNotebook 03 completed successfully.")

NOTEBOOK 03 VALIDATION
Original cleaned dataset: (49860, 36)
Feature-engineered dataset: (49860, 41)
Training rows: 39,888
Testing rows: 9,972
Input model features before encoding: 40
Final encoded features: 102
Training acceptance rate: 63.85%
Testing acceptance rate: 63.85%
Train/test overlap: 0
Training matrix contains NaN: False
Test matrix contains NaN: False

Notebook 03 completed successfully.
